<a href="https://colab.research.google.com/github/felixyustian/enterprise_ai_context_engine_fraud_risk_marketing/blob/main/enterprise_ai_context_engine_fraud_risk_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL 1] Instalasi MCP SDK & Enterprise AI Stack
!pip install -qU mcp langchain langchain-google-genai langchain-community langgraph faiss-cpu pandas streamlit
!npm install -q -g localtunnel

print("✅ Infrastruktur MCP & AI Engine berhasil disiapkan!")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧✅ Infrastruktur MCP & AI Engine berhasil disiapkan!


In [ ]:
%%writefile engine.py
import json
import pandas as pd
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# --- MCP SERVER: External Credit Bureau (Simulasi SLIK OJK/Pefindo) ---
class CreditBureauMCP:
    def __init__(self):
        # Database simulasi riwayat kredit calon debitur
        self.credit_db = pd.DataFrame({
            "app_id": ["APP-501", "APP-502", "APP-503"],
            "applicant_name": ["Budi Santoso", "Siti Aminah", "PT. Teknologi Bangsa"],
            "credit_score": [720, 580, 810],
            "debt_to_income_ratio": [0.25, 0.65, 0.15],
            "late_payments_24m": [0, 4, 0],
            "requested_amount_idr": [150000000, 50000000, 2000000000]
        })

    def call_tool(self, tool_name: str, arguments: dict):
        if tool_name == "get_credit_profile":
            app_id = arguments.get("app_id", "")
            data = self.credit_db[self.credit_db['app_id'] == app_id].to_dict('records')
            return json.dumps(data[0] if data else {"error": "Applicant Not Found in Bureau"})
        return json.dumps({"error": "Unknown tool"})

# --- AGENTIC CORE: Credit Underwriter ---
class UnderwritingState(TypedDict):
    app_id: str
    credit_data: str
    final_decision: str

def init_underwriting_engine(api_key: str):
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1,
        api_key=api_key
    )

    mcp_server = CreditBureauMCP()

    # Node 1: Credit Data Fetcher (MCP Client)
    def fetch_credit_node(state: UnderwritingState) -> dict:
        print(f"🔗 [MCP] Menarik profil kredit untuk aplikasi {state['app_id']}...")
        raw_data = mcp_server.call_tool("get_credit_profile", {"app_id": state["app_id"]})
        return {"credit_data": raw_data}

    # Node 2: AI Underwriting Analyst
    def underwriting_node(state: UnderwritingState) -> dict:
        prompt = f"""
        Anda adalah AI Chief Credit Underwriter di sebuah Bank.

        Data Profil Kredit via MCP SLIK OJK: {state['credit_data']}

        Instruksi Evaluasi:
        1. Analisis kelayakan kredit berdasarkan 'credit_score' (Aman > 700), 'debt_to_income_ratio' (Maks 0.40), dan 'late_payments_24m' (Maks 1).
        2. Berikan Keputusan Akhir dalam format tag: [APPROVED], [REJECTED], atau [MANUAL_REVIEW].
        3. Susun alasan persetujuan/penolakan dalam poin-poin yang profesional dan berikan syarat tambahan jika berstatus MANUAL_REVIEW.
        """
        response = llm.invoke(prompt)
        return {"final_decision": response.content}

    # Merakit Graph
    workflow = StateGraph(UnderwritingState)
    workflow.add_node("Data_Fetcher", fetch_credit_node)
    workflow.add_node("Underwriter", underwriting_node)

    workflow.add_edge(START, "Data_Fetcher")
    workflow.add_edge("Data_Fetcher", "Underwriter")
    workflow.add_edge("Underwriter", END)

    return workflow.compile()

def run_credit_analysis(app_id: str, api_key: str):
    app = init_underwriting_engine(api_key)
    return app.invoke({"app_id": app_id})

Overwriting engine.py


In [ ]:
%%writefile app.py
import streamlit as st
import time
import json

# 1. MENGUBAH JUDUL WEB PAGE (Muncul di Tab Browser)
st.set_page_config(
    page_title="Enterprise AI Context Engine: Fraud, Risk & Marketing Analytics",
    page_icon="⚡",
    layout="wide"
)

@st.cache_resource(show_spinner="⏳ Memuat AI Engine ke Memory... (Hanya 1x)")
def get_ai_app(api_key):
    import engine
    return engine.init_underwriting_engine(api_key)

# 2. MENGUBAH JUDUL HEADER (Muncul di dalam halaman aplikasi)
st.title("⚡ Fraud, Risk & Marketing Analytics Engine")
st.markdown("Arsitektur: **LangGraph + MCP** | Status: **High-Performance Cached**")
st.divider()

if "api_key_valid" not in st.session_state:
    st.session_state.api_key_valid = False

with st.sidebar:
    st.header("🔑 Authentication")
    input_key = st.text_input("Masukkan Google API key anda", type="password")
    if st.button("Activate Engine"):
        if input_key:
            st.session_state.api_key_valid = True
            st.success("Engine Active!")
        else:
            st.error("Masukkan API Key")

if st.session_state.api_key_valid:
    col1, col2 = st.columns([1, 2])

    with col1:
        user_query = st.text_input("Masukkan ID Aplikasi / Kueri:", value="APP-502")
        execute = st.button("Run Analytics 🚀")

    if execute:
        start_time = time.time()

        with st.status("🚀 Processing...", expanded=True) as status:
            st.write("🔧 Menyiapkan Engine...")
            app_instance = get_ai_app(input_key)

            st.write("🤖 AI Agent sedang bekerja...")
            result = app_instance.invoke({"app_id": user_query})

            status.update(label="✅ Analysis Selesai!", state="complete", expanded=False)

        duration = time.time() - start_time

        with col2:
            st.subheader("Final Strategic Report")
            st.caption(f"Waktu Eksekusi: {duration:.2f} detik")
            st.markdown(result.get('final_decision') or result.get('final_report') or result.get('aml_report'))
else:
    st.info("👈 Masukkan API Key Anda di sebelah kiri agar antarmuka AI muncul seketika.")

Overwriting app.py


In [ ]:
# =====================================================================
# [CELL 4] SERVER DEPLOYMENT: STREAMLIT WEBSOCKET TUNING
# =====================================================================
import time
import subprocess
from google.colab import output

print("🧹 1. Membersihkan jalur komunikasi...")
!pkill -9 -f streamlit
!fuser -k 8502/tcp
time.sleep(2)

print("🚀 2. Memulai Streamlit dengan WebSocket Compression...")
# Menambahkan argumen khusus untuk menstabilkan koneksi via Proxy Google
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8502",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "true", # Kunci untuk melancarkan loading
    "--browser.gatherUsageStats", "false"          # Mematikan telemetri yang bikin lambat
])

print("⏳ Menyiapkan infrastruktur jaringan (5 detik)...")
time.sleep(5)

print("🛡️ 3. Membuka UI di tab baru...")
output.serve_kernel_port_as_window(8502)

🚀 1. Memulai Engine Streamlit Backend...
🛡️ 2. Membangun jalur aman via infrastruktur internal Google...
👉 Tautan aman Anda sedang diproses oleh Google...

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>